## Подход к решению

### Выбор моделей

Данные представлены таблицей с числовыми счетчиками, категориальными признаками, пропусками и нелинейными зависимостями. Размер обучающей выборки составляет около четырнадцати тысяч строк. Для такого сочетания размера и структуры данных выбраны модели градиентного бустинга по деревьям.

Используются CatBoost, LightGBM и XGBoost. CatBoost обрабатывает категории без предварительного one hot кодирования. LightGBM строит деревья другим способом. XGBoost использует другой набор регуляризаторов. Различия между алгоритмами создают разнообразие прогнозов для ансамбля.

Нейронные сети не используются из-за размера выборки и структуры признаков. 

### Работа с целевой метрикой

Модели обучаются как бинарные классификаторы на Logloss. Параметры и число итераций выбираются по Daily Average Precision на временных разбиениях. Перед ансамблированием прогнозы преобразуются в относительные ранги внутри даты.

Доля положительного класса составляет около двадцати процентов. Искусственное увеличение или уменьшение классов не применяется, поскольку оно изменило бы временное распределение данных. Компенсация дисбаланса выполняется через `scale_pos_weight`, который подбирается Optuna.

### Временная валидация

Тестовый период расположен после обучающего периода. Случайное разбиение позволило бы модели использовать закономерности из будущих дат. Поэтому применяется скользящее временное окно. В каждом разбиении все обучающие даты находятся раньше валидационной даты.

Наблюдения получают равный вклад по дням. Дополнительное экспоненциальное взвешивание усиливает недавние даты и учитывает изменение поведения между началом обучения и тестовым периодом.

### Генерация признаков

События фильтруются условием `event_ts < assignment_ts` до расчета любых агрегатов.

Из событий рассчитываются количество и давность действий, активность в окнах, затухающие суммы, интервалы между событиями, сессии, последние последовательности, переходы и этапы воронки. Из готовых табличных агрегатов рассчитываются скорости, доли и изменения между временными окнами.

Идентификаторы `lead_id` и `user_id` не передаются моделям. Они используются только для соединения таблиц и формирования результата, т.к они не несут в себе информации из-за нулевого пересечения train и test.

### Модель на недавнем периоде

`cat_recent` использует ту же структуру CatBoost, но обучается только на последних десяти днях каждого обучающего периода. Ее задача состоит в снижении влияния старых наблюдений.

Для `cat_recent` используются параметры основной CatBoost-модели, отдельный подбор привёл бы к слишком большим временным затратам на обучение даже с ресурсами кеггла.

### Настройка и ансамбль

Optuna подбирает параметры структуры деревьев, регуляризации, доли объектов, доли признаков, скорости обучения и `scale_pos_weight`.

Веса ансамбля подбираются только по прогнозам вне выборки. Используются дневные ранги моделей.

Финальные модели обучаются на всей обучающей выборке с пятью фиксированными сидами. Усреднение по сидам снижает зависимость результата от случайности обучения.

CatBoost обучается в режиме Plain, а не Ordered, т.к. времени на тренировку Ordered не хватило бы даже с Kaggle.

Ноутбук запускался и предназначается для запуска на Kaggle с использованием датасета, содержащего все необходимые файлы. Для получения результата, идентичного моему, необходимо сохранить все параметры и запустить ноутбук на Kaggle с ускорителем GPU T4 x 2.

## Среда Kaggle

Исходные файлы загружаются из подключенного Kaggle Dataset. Результаты сохраняются в каталог `/kaggle/working/lead-prioritization-output`.


In [ ]:

from __future__ import annotations

import gc
import hashlib
import json
import logging
import math
import os
import shutil
import subprocess
import re
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier, Pool
import lightgbm as lgb
from optuna.samplers import TPESampler
import xgboost as xgb

# Предупреждения библиотек скрываются, чтобы журнал обучения содержал только сообщения решения
warnings.filterwarnings("ignore")
# Optuna выводит только предупреждения, а ход поиска записывается собственным журналом
optuna.logging.set_verbosity(optuna.logging.WARNING)

TARGET = "target"
RANDOM_STATE = 2501
ID_COLUMNS = {"lead_id", "user_id"}
TIME_COLUMNS = {"assignment_ts", "assignment_date"}
# Идентификаторы, целевая переменная и исходные временные метки не передаются моделям напрямую
NON_FEATURE_COLUMNS = ID_COLUMNS | TIME_COLUMNS | {TARGET, "split"}
MISSING_CATEGORY = "__MISSING__"
CPU_THREADS = max(1, min(8, os.cpu_count() or 1))
# Единый порядок окон нужен для расчета скоростей и изменений между соседними периодами
WINDOW_DAYS = {"1d": 1.0, "3d": 3.0, "7d": 7.0, "14d": 14.0, "30d": 30.0, "90d": 90.0}
WINDOW_ORDER = ["1d", "3d", "7d", "14d", "30d", "90d"]
EVENT_WINDOWS_SECONDS = {
    "5m": 5 * 60.0,
    "15m": 15 * 60.0,
    "30m": 30 * 60.0,
    "1h": 3600.0,
    "2h": 2 * 3600.0,
    "3h": 3 * 3600.0,
    "6h": 6 * 3600.0,
    "12h": 12 * 3600.0,
    "1d": 86400.0,
    "3d": 3 * 86400.0,
    "7d": 7 * 86400.0,
    "14d": 14 * 86400.0,
    "30d": 30 * 86400.0,
}


# Неизменяемая конфигурация хранит все параметры, влияющие на воспроизводимость результата
@dataclass(frozen=True)
class RunConfig:
    data_dir: str
    output_dir: str
    cv_folds: int = 6
    tune_folds: int = 5
    valid_days: int = 1
    min_train_days: int = 10
    tune_trials: int = 40
    max_rounds: int = 2400
    early_stopping_rounds: int = 150
    recent_days: int = 10
    recency_half_life_days: float = 7.0
    final_seeds: int = 5
    blend_trials: int = 15000
    blend_shrinkage: float = 0.06


# Проверяет доступность видеокарты NVIDIA
def has_nvidia_gpu() -> bool:
    if shutil.which("nvidia-smi") is None:
        return False
    result = subprocess.run(
        ["nvidia-smi", "-L"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
        timeout=10,
    )
    return result.returncode == 0


GPU_AVAILABLE = has_nvidia_gpu()
CAT_DEVICE = "GPU" if GPU_AVAILABLE else "CPU"
MODEL_DEVICES = {
    "lgb": "cpu",
    "xgb": "cuda" if GPU_AVAILABLE else "cpu",
    "cat": CAT_DEVICE,
    "cat_recent": CAT_DEVICE,
}
CONFIG = RunConfig(
    data_dir="/kaggle/input/datasets/danegumya/lead-prioritization-data",
    output_dir="/kaggle/working/lead-prioritization-output",
)


## Загрузка и проверка данных


In [ ]:
LOGGER = logging.getLogger("lead_priority")


# Настраивает вывод журнала в консоль и файл
def configure_logging(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    formatter = logging.Formatter("%(asctime)sZ | %(levelname)s | %(message)s", "%H:%M:%S")
    formatter.converter = time.gmtime
    LOGGER.setLevel(logging.INFO)
    LOGGER.handlers.clear()
    console = logging.StreamHandler(sys.stdout)
    console.setFormatter(formatter)
    file_handler = logging.FileHandler(output_dir / "training.log", encoding="utf-8")
    file_handler.setFormatter(formatter)
    LOGGER.addHandler(console)
    LOGGER.addHandler(file_handler)


# Возвращает максимальный объем памяти процесса в гигабайтах
def memory_gb() -> float:
    try:
        import resource

        value = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        # macOS возвращает байты, а Linux возвращает килобайты
        divisor_power = 3 if sys.platform == "darwin" else 2
        return value / 1024.0 ** divisor_power
    except Exception:
        return float("nan")


# Удаляет неиспользуемые объекты и записывает использование памяти
def collect_garbage(stage: str) -> None:
    collected = gc.collect()
    LOGGER.info("GC %-24s objects=%d peak_ram=%.2fGB", stage, collected, memory_gb())


# Загружает файлы из подключенного набора данных
# Загружает обучающую, тестовую, событийную таблицы и пример отправки
def load_data(data_dir: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    root = Path(data_dir)
    train = pd.read_csv(root / "train.csv", low_memory=False)
    test = pd.read_csv(root / "test.csv", low_memory=False)
    events = pd.read_csv(root / "events.csv", low_memory=False)
    sample = pd.read_csv(root / "sample_submission.csv", low_memory=False)
    return train, test, events, sample



# Проверяет обязательные колонки, идентификаторы и целевую переменную
def validate_input(train: pd.DataFrame, test: pd.DataFrame, events: pd.DataFrame) -> None:
    required_train = {"lead_id", "user_id", "assignment_ts", "assignment_date", TARGET}
    required_test = required_train - {TARGET}
    required_events = {"lead_id", "user_id", "event_ts", "event_type"}
    missing_train = required_train - set(train.columns)
    missing_test = required_test - set(test.columns)
    missing_events = required_events - set(events.columns)
    if missing_train or missing_test or missing_events:
        raise ValueError(
            f"Отсутствуют колонки. В обучении: {sorted(missing_train)} "
            f"В тесте: {sorted(missing_test)} В событиях: {sorted(missing_events)}"
        )
    # Уникальность обращения необходима для последующего объединения
    if not train["lead_id"].is_unique or not test["lead_id"].is_unique:
        raise ValueError("Идентификатор lead_id должен быть уникальным")
    if set(train["lead_id"]) & set(test["lead_id"]):
        raise ValueError("Пересечение идентификаторов lead_id между обучающей и тестовой выборками")
    if not set(train[TARGET].dropna().unique()).issubset({0, 1}):
        raise ValueError("Целевая переменная должна содержать только бинарные значения")


# Преобразует значение в безопасную часть имени признака
def sanitize_token(value: Any) -> str:
    token = re.sub(r"[^0-9a-zA-Z_]+", "_", str(value).strip().lower())
    token = re.sub(r"_+", "_", token).strip("_")
    return token or "unknown"


# Уменьшает разрядность числовых колонок для экономии памяти
def reduce_numeric_memory(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="float")
    for col in df.select_dtypes(include=["int64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    return df


# Добавляет префикс к именам колонок агрегированной таблицы
def _prefix_columns(frame: pd.DataFrame, prefix: str) -> pd.DataFrame:
    frame = frame.copy()
    frame.columns = [f"{prefix}{sanitize_token(c)}" for c in frame.columns]
    return frame


## Анализ данных


In [ ]:
# Описывает размеры таблиц и временные границы выборок
def analyze_general_bounds(train: pd.DataFrame, test: pd.DataFrame, events: pd.DataFrame) -> list[str]:
    return [
        "Размерности данных:",
        f"Обучение: {train.shape}",
        f"Тест: {test.shape}",
        f"События: {events.shape}",
        "",
        "Временные границы:",
        f"Обучение: {train['assignment_ts'].min()} — {train['assignment_ts'].max()}",
        f"Тест: {test['assignment_ts'].min()} — {test['assignment_ts'].max()}",
        f"События: {events['event_ts'].min()} — {events['event_ts'].max()}"
    ]


# Считает общую и дневную долю положительного класса
def analyze_target(train: pd.DataFrame) -> list[str]:
    target_mean = train["target"].mean()
    train["assignment_date_temp"] = train["assignment_ts"].dt.date
    daily_target = train.groupby("assignment_date_temp")["target"].agg(["count", "mean"])
    train.drop(columns=["assignment_date_temp"], inplace=True)
    return [
        f"Общее среднее целевой переменной: {target_mean:.4f}",
        "Среднее значение по дням:",
        daily_target.to_string()
    ]


# Проверяет уникальность обращений и пересечение пользователей
def analyze_identifiers(train: pd.DataFrame, test: pd.DataFrame) -> list[str]:
    train_users = set(train["user_id"].dropna().unique())
    test_users = set(test["user_id"].dropna().unique())
    intersection = len(train_users.intersection(test_users))
    return [
        f"Уникальные обращения обучающей выборки: {train['lead_id'].nunique()} из {len(train)}",
        f"Уникальные обращения тестовой выборки: {test['lead_id'].nunique()} из {len(test)}",
        f"Уникальные пользователи обучающей выборки: {len(train_users)}",
        f"Уникальные пользователи тестовой выборки: {len(test_users)}",
        f"Пересечение пользователей между обучением и тестом: {intersection}"
    ]


# Измеряет долю событий после назначения и объем допустимой истории
def analyze_events_leakage(train: pd.DataFrame, test: pd.DataFrame, events: pd.DataFrame) -> list[str]:
    all_assignments = pd.concat([
        train[["lead_id", "assignment_ts"]], test[["lead_id", "assignment_ts"]]
    ]).drop_duplicates(subset=["lead_id"])

    ev_merged = events.merge(all_assignments, on="lead_id", how="inner")
    # Положительное значение означает, что событие уже было известно в момент скоринга
    ev_merged["time_to_assignment_sec"] = (
        ev_merged["assignment_ts"] - ev_merged["event_ts"]
    ).dt.total_seconds()

    future_events = (ev_merged["time_to_assignment_sec"] < 0).mean() * 100
    valid_ev = ev_merged[ev_merged["time_to_assignment_sec"] >= 0]
    events_per_lead = valid_ev.groupby("lead_id").size()

    return [
        f"ВНИМАНИЕ: {future_events:.2f}% событий произошли после момента назначения",
        "Время до назначения для корректных событий в секундах:",
        valid_ev["time_to_assignment_sec"].describe(
            percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]
        ).to_string(),
        "Количество корректных событий на одно обращение:",
        events_per_lead.describe(
            percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]
        ).to_string()
    ]


# Формирует и выводит сводный отчет анализа
def run_analysis(train: pd.DataFrame, test: pd.DataFrame, events: pd.DataFrame) -> None:
    # Копии защищают исходные таблицы от временных колонок и преобразования дат
    t_train = train.copy()
    t_test = test.copy()
    t_events = events.copy()
    t_train["assignment_ts"] = pd.to_datetime(t_train["assignment_ts"], errors="coerce")
    t_test["assignment_ts"] = pd.to_datetime(t_test["assignment_ts"], errors="coerce")
    t_events["event_ts"] = pd.to_datetime(t_events["event_ts"], errors="coerce")

    report = []
    report.extend(["=" * 60, "1. ОБЩИЕ РАЗМЕРНОСТИ И ВРЕМЯ", "=" * 60])
    report.extend(analyze_general_bounds(t_train, t_test, t_events))
    report.extend(["\n" + "=" * 60, "2. РАСПРЕДЕЛЕНИЕ ЦЕЛЕВОЙ ПЕРЕМЕННОЙ", "=" * 60])
    report.extend(analyze_target(t_train))
    report.extend(["\n" + "=" * 60, "3. ИДЕНТИФИКАТОРЫ И ПЕРЕСЕЧЕНИЯ", "=" * 60])
    report.extend(analyze_identifiers(t_train, t_test))
    report.extend(["\n" + "=" * 60, "4. АНАЛИЗ СОБЫТИЙ И УТЕЧЕК ДАННЫХ", "=" * 60])
    report.extend(analyze_events_leakage(t_train, t_test, t_events))

    print("\n".join(report))


In [4]:
train_raw, test_raw, events_raw, sample_submission_raw = load_data(CONFIG.data_dir)
validate_input(train_raw, test_raw, events_raw)
run_analysis(train_raw, test_raw, events_raw)
# После анализа исходные таблицы удаляются, поскольку полный цикл загрузит их повторно
del train_raw, test_raw, events_raw, sample_submission_raw
gc.collect()


1. ОБЩИЕ РАЗМЕРНОСТИ И ВРЕМЯ
Размерности данных:
Обучение: (13694, 119)
Тест: (4306, 118)
События: (254705, 7)

Временные границы:
Обучение: 2026-04-07 09:02:00 — 2026-04-22 21:58:00
Тест: 2026-04-23 09:00:00 — 2026-04-27 21:59:00
События: 2026-03-08 10:03:00 — 2026-04-27 20:28:00

2. РАСПРЕДЕЛЕНИЕ ЦЕЛЕВОЙ ПЕРЕМЕННОЙ
Общее среднее целевой переменной: 0.2075
Среднее значение по дням:
                      count      mean
assignment_date_temp                 
2026-04-07              873  0.223368
2026-04-08              832  0.209135
2026-04-09              859  0.206054
2026-04-10              830  0.220482
2026-04-11              861  0.203252
2026-04-12              855  0.203509
2026-04-13              791  0.206068
2026-04-14              842  0.197150
2026-04-15              899  0.199110
2026-04-16              854  0.203747
2026-04-17              901  0.221976
2026-04-18              875  0.203429
2026-04-19              863  0.198146
2026-04-20              861  0.191638
2026-0

79

## Выводы по анализу данных

Пересечение идентификаторов пользователей между обучающей и тестовой выборками равно нулю. Идентификатор `user_id` исключается из признаков. Модель использует обобщенные поведенческие характеристики, которые применимы к ранее не встречавшимся пользователям.

Целевая переменная имеет стабильное среднее значение около двадцати процентов. Колебания показателя по дням не требуют искусственного изменения обучающей выборки. Компенсация дисбаланса выполняется путем подбора `scale_pos_weight` отдельно для каждой модели.

Исследование журнала дополнительных действий выявило утечку данных. Более шести процентов записей имеют временную метку позже времени назначения обращения. Код Feature Engineering-а отбрасывает любые действия с `event_ts >= assignment_ts` до расчета агрегатов.



## Признаки на основе событий


In [ ]:
# Строит признаки из событий, доступных строго до назначения обращения
def build_event_features(
        train: pd.DataFrame,
        test: pd.DataFrame,
        events: pd.DataFrame,
) -> pd.DataFrame:
    # Связываем события с точным моментом назначения обращения
    # Блок подготовки связывает журнал с назначениями и задает допустимую временную область
    assignment_cols = ["lead_id", "user_id", "assignment_ts"]
    if "item_price_log" in train.columns and "item_price_log" in test.columns:
        assignment_cols.append("item_price_log")

    assignments = pd.concat(
        [train[assignment_cols], test[assignment_cols]],
        ignore_index=True,
    ).copy()
    assignments["assignment_ts"] = pd.to_datetime(assignments["assignment_ts"], errors="coerce")
    assignments = assignments.rename(columns={"user_id": "assignment_user_id"})
    if "item_price_log" in assignments.columns:
        assignments = assignments.rename(columns={"item_price_log": "assignment_item_price_log"})

    ev = events.merge(assignments, on="lead_id", how="inner", validate="many_to_one")
    ev["event_ts"] = pd.to_datetime(ev["event_ts"], errors="coerce")

    # Исключаем события, которые произошли после создания заявки
    ev = ev[
        ev["event_ts"].notna()
        & ev["assignment_ts"].notna()
        & (ev["event_ts"] < ev["assignment_ts"])
        ].copy()

    # Событие учитывается только при совпадении пользователя события и пользователя назначения
    if "user_id" in ev.columns:
        same_user = ev["user_id"].astype(str) == ev["assignment_user_id"].astype(str)
        ev = ev[same_user].copy()

    # При отсутствии допустимой истории возвращается минимальная таблица для корректного merge-а
    if ev.empty:
        return assignments[["lead_id"]].drop_duplicates().assign(event_count_all=0.0)

    ev["event_type_clean"] = ev["event_type"].map(sanitize_token)
    ev["age_sec"] = (ev["assignment_ts"] - ev["event_ts"]).dt.total_seconds()
    ev["age_days"] = ev["age_sec"] / 86400.0
    ev["event_hour"] = ev["event_ts"].dt.hour.astype("float32")
    ev["event_weekday"] = ev["event_ts"].dt.weekday.astype("float32")
    # фиксируем порядок равных временных меток для последовательных признаков
    ev = ev.sort_values(["lead_id", "event_ts"], kind="mergesort").reset_index(drop=True)

    # Блок базовых агрегатов собирает частоту, давность и распределение типов событий
    feature_frames: list[pd.DataFrame | pd.Series] = []

    # Считаем объем истории, разнообразие действий и давность активности
    overall = ev.groupby("lead_id", observed=True).agg(
        event_count_all=("event_type_clean", "size"),
        event_type_nunique=("event_type_clean", "nunique"),
        event_recency_sec=("age_sec", "min"),
        event_oldest_sec=("age_sec", "max"),
        event_active_span_sec=("event_ts", lambda s: (s.max() - s.min()).total_seconds()),
    )
    feature_frames.append(overall)

    # Разворот по типу события создает отдельный счетчик каждого действия
    type_counts = ev.groupby(["lead_id", "event_type_clean"], observed=True).size().unstack(fill_value=0)
    type_counts = _prefix_columns(type_counts, "event_count_type_")
    feature_frames.append(type_counts)

    # Минимальный возраст соответствует самому свежему событию данного типа
    type_recency = ev.groupby(["lead_id", "event_type_clean"], observed=True)["age_sec"].min().unstack()
    type_recency = _prefix_columns(type_recency, "event_recency_type_")
    feature_frames.append(type_recency)

    # Считаем интенсивность всех типов действий в коротких и длинных окнах
    for window_name, max_age in EVENT_WINDOWS_SECONDS.items():
        part = ev[ev["age_sec"] <= max_age]
        count_all = part.groupby("lead_id", observed=True).size().rename(f"event_count_{window_name}")
        feature_frames.append(count_all)
        if not part.empty:
            by_type = part.groupby(["lead_id", "event_type_clean"], observed=True).size().unstack(fill_value=0)
            by_type = _prefix_columns(by_type, f"event_count_{window_name}_type_")
            feature_frames.append(by_type)

    # Придаем больший вес недавним событиям с помощью экспоненциального затухания
    for half_life_days in [1.0, 3.0, 7.0, 14.0, 30.0]:
        name = str(int(half_life_days))
        weight_col = f"_decay_{name}d"
        ev[weight_col] = np.exp(-ev["age_days"] * math.log(2.0) / half_life_days).astype("float32")
        decay_all = ev.groupby("lead_id", observed=True)[weight_col].sum().rename(f"event_decay_{name}d")
        feature_frames.append(decay_all)
        decay_type = ev.groupby(["lead_id", "event_type_clean"], observed=True)[weight_col].sum().unstack(fill_value=0)
        decay_type = _prefix_columns(decay_type, f"event_decay_{name}d_type_")
        feature_frames.append(decay_type)

    # Считаем распределения контекста события и его сочетания с типом действия
    for categorical_event_col in ["ctx_seq", "src_slot"]:
        if categorical_event_col not in ev.columns:
            continue
        clean_col = f"_{categorical_event_col}_clean"
        ev[clean_col] = ev[categorical_event_col].map(sanitize_token)
        value_counts = ev.groupby(["lead_id", clean_col], observed=True).size().unstack(fill_value=0)
        value_counts = _prefix_columns(value_counts, f"event_{categorical_event_col}_count_")
        feature_frames.append(value_counts)
        interaction = (
            ev["event_type_clean"].astype(str) + "__" + ev[clean_col].astype(str)
        ).rename("_event_context")
        interaction_frame = pd.DataFrame({"lead_id": ev["lead_id"], "value": interaction})
        interaction_counts = interaction_frame.groupby(
            ["lead_id", "value"], observed=True
        ).size().unstack(fill_value=0)
        interaction_counts = _prefix_columns(
            interaction_counts, f"event_type_{categorical_event_col}_count_"
        )
        feature_frames.append(interaction_counts)

    # Описываем регулярность активности через интервалы между соседними событиями
    # Регулярность действий и пользовательские сессии
    ev["gap_sec"] = ev.groupby("lead_id", observed=True)["event_ts"].diff().dt.total_seconds()
    gap_features = ev.groupby("lead_id", observed=True)["gap_sec"].agg(
        event_gap_mean_sec="mean",
        event_gap_median_sec="median",
        event_gap_std_sec="std",
        event_gap_min_sec="min",
        event_gap_max_sec="max",
    )
    feature_frames.append(gap_features)

    # Разделяем действия на сессии при паузе между ними более тридцати минут
    ev["new_session"] = ev["gap_sec"].isna() | (ev["gap_sec"] > 1800.0)
    ev["session_id"] = ev.groupby("lead_id", observed=True)["new_session"].cumsum().astype("int32")
    session = ev.groupby(["lead_id", "session_id"], observed=True).agg(
        session_events=("event_type_clean", "size"),
        session_start=("event_ts", "min"),
        session_end=("event_ts", "max"),
    )
    session["session_duration_sec"] = (session["session_end"] - session["session_start"]).dt.total_seconds()
    session_features = session.groupby(level=0, observed=True).agg(
        event_session_count=("session_events", "size"),
        event_session_events_mean=("session_events", "mean"),
        event_session_events_max=("session_events", "max"),
        event_session_duration_mean_sec=("session_duration_sec", "mean"),
        event_session_duration_max_sec=("session_duration_sec", "max"),
    )
    feature_frames.append(session_features)

    # Считаем агрегаты цены и позиции источника внутри истории обращения
    numeric_aggs: dict[str, tuple[str, str]] = {}
    if "item_price_log" in ev.columns:
        numeric_aggs.update(
            event_price_mean=("item_price_log", "mean"),
            event_price_std=("item_price_log", "std"),
            event_price_min=("item_price_log", "min"),
            event_price_max=("item_price_log", "max"),
        )
    if "src_slot" in ev.columns:
        numeric_aggs.update(
            event_src_slot_nunique=("src_slot", "nunique"),
            event_src_slot_mean=("src_slot", "mean"),
            event_src_slot_std=("src_slot", "std"),
            event_src_slot_min=("src_slot", "min"),
            event_src_slot_max=("src_slot", "max"),
        )
    if "ctx_seq" in ev.columns:
        numeric_aggs["event_ctx_seq_nunique"] = ("ctx_seq", "nunique")
    if numeric_aggs:
        feature_frames.append(ev.groupby("lead_id", observed=True).agg(**numeric_aggs))

    # Сохраняем первое действие, последние пять действий и путь пользователя
    # Восстанавливает начало, конец и последние шаги истории
    last = ev.groupby("lead_id", observed=True).tail(1).set_index("lead_id")
    first = ev.groupby("lead_id", observed=True).head(1).set_index("lead_id")
    sequence_features = pd.DataFrame(index=last.index)
    sequence_features["event_last_type"] = last["event_type_clean"].astype(str)
    sequence_features["event_first_type"] = first["event_type_clean"].reindex(last.index).astype(str)

    ev["reverse_event_position"] = ev.groupby("lead_id", observed=True).cumcount(ascending=False)
    last_five = ev.loc[
        ev["reverse_event_position"] < 5,
        ["lead_id", "reverse_event_position", "event_type_clean", "age_sec"],
    ]
    type_last_wide = last_five.pivot(index="lead_id", columns="reverse_event_position", values="event_type_clean")
    age_last_wide = last_five.pivot(index="lead_id", columns="reverse_event_position", values="age_sec")
    for position in range(5):
        if position in type_last_wide.columns:
            sequence_features[f"event_type_last_{position + 1}"] = type_last_wide[position].reindex(
                sequence_features.index).astype("string")
            sequence_features[f"event_age_sec_last_{position + 1}"] = age_last_wide[position].reindex(
                sequence_features.index
            )
    sequence_features["event_last3_path"] = (
        sequence_features.reindex(columns=["event_type_last_3", "event_type_last_2", "event_type_last_1"])
        .fillna("none")
        .astype(str)
        .agg(">".join, axis=1)
    )
    sequence_features["event_last5_path"] = (
        sequence_features.reindex(
            columns=[f"event_type_last_{i}" for i in [5, 4, 3, 2, 1]]
        ).fillna("none").astype(str).agg(">".join, axis=1)
    )
    sequence_features["event_last_hour"] = last["event_hour"]
    sequence_features["event_last_weekday"] = last["event_weekday"]

    if "ctx_seq" in ev.columns:
        sequence_features["event_last_ctx_seq"] = last["ctx_seq"].astype(str)
    if "src_slot" in ev.columns:
        sequence_features["event_last_src_slot_cat"] = last["src_slot"].astype(str)
        sequence_features["event_first_src_slot_cat"] = first["src_slot"].reindex(last.index).astype(str)
        sequence_features["event_src_slot_last_minus_first"] = (
            pd.to_numeric(last["src_slot"], errors="coerce")
            - pd.to_numeric(first["src_slot"].reindex(last.index), errors="coerce")
        )
    if "item_price_log" in ev.columns:
        sequence_features["event_last_price_log"] = last["item_price_log"]
    feature_frames.append(sequence_features)

    # Переводим действия в этапы воронки и измеряем движение к контакту
    # События превращам в упорядоченные этапы и считает направление движения
    stage_map = {"search": 0.0, "item_view": 1.0, "favorite": 2.0, "chat_open": 3.0, "call_click": 4.0}
    ev["event_stage"] = ev["event_type_clean"].map(stage_map).fillna(0.0).astype("float32")
    ev["event_stage_diff"] = ev.groupby("lead_id", observed=True)["event_stage"].diff()
    stage_features = ev.groupby("lead_id", observed=True).agg(
        event_stage_mean=("event_stage", "mean"),
        event_stage_max=("event_stage", "max"),
        event_stage_last=("event_stage", "last"),
        event_stage_progressions=("event_stage_diff", lambda x: float((x > 0).sum())),
        event_stage_regressions=("event_stage_diff", lambda x: float((x < 0).sum())),
        event_high_intent_count=("event_stage", lambda x: float((x >= 2).sum())),
    )
    feature_frames.append(stage_features)

    for half_life_days in [1.0, 3.0, 7.0]:
        weight = np.exp(-ev["age_days"] * math.log(2.0) / half_life_days)
        feature_frames.append(
            (ev["event_stage"] * weight).groupby(ev["lead_id"], observed=True).sum()
            .rename(f"event_stage_decay_{int(half_life_days)}d")
        )

    # Считаем переходы между соседними типами событий
    ev["previous_event_type"] = ev.groupby("lead_id", observed=True)["event_type_clean"].shift(1)
    transition_rows = ev[ev["previous_event_type"].notna()].copy()
    if not transition_rows.empty:
        transition_rows["event_transition"] = (
                transition_rows["previous_event_type"].astype(str)
                + "__to__"
                + transition_rows["event_type_clean"].astype(str)
        )
        transition_counts = transition_rows.groupby(
            ["lead_id", "event_transition"], observed=True
        ).size().unstack(fill_value=0)
        transition_counts = _prefix_columns(transition_counts, "event_transition_count_")
        feature_frames.append(transition_counts)

    # Энтропия отражает разнообразие типов действий пользователя
    # Счетчики нормируются в вероятности перед вычислением энтропии Шеннона
    count_matrix = type_counts.astype("float64")
    row_sum = count_matrix.sum(axis=1).replace(0, np.nan)
    probabilities = count_matrix.div(row_sum, axis=0)
    entropy = -(probabilities * np.log(probabilities.where(probabilities > 0))).sum(axis=1)
    feature_frames.append(entropy.rename("event_type_entropy"))

    # Все агрегаты объединяются за одну операцию, чтобы избежать фрагментации DF-а
    # Объединение агрегатов, заполнение счетчиков и построение отношений
    features = pd.concat(feature_frames, axis=1)

    count_like_prefixes = (
        "event_count_",
        "event_decay_",
        "event_transition_count_",
        "event_ctx_seq_count_",
        "event_src_slot_count_",
        "event_type_ctx_seq_count_",
        "event_type_src_slot_count_",
        "event_session_count",
        "event_type_nunique",
        "event_src_slot_nunique",
        "event_ctx_seq_nunique",
        "event_stage_progressions",
        "event_stage_regressions",
        "event_high_intent_count",
    )
    # Пропуск у счетчика означает отсутствие события, поэтому его можно заменить нулем
    for col in features.columns:
        if col.startswith(count_like_prefixes):
            features[col] = features[col].fillna(0.0)

    # Отношения контактов к просмотрам описывают глубину намерения
    ratio_specs = [
        ("call_click", "item_view", "event_call_to_view_ratio"),
        ("chat_open", "item_view", "event_chat_to_view_ratio"),
        ("favorite", "item_view", "event_favorite_to_view_ratio"),
    ]
    for numerator, denominator, out_name in ratio_specs:
        num_col = f"event_count_type_{numerator}"
        den_col = f"event_count_type_{denominator}"
        if num_col in features.columns and den_col in features.columns:
            features[out_name] = features[num_col] / (features[den_col] + 1.0)

    # Доли коротких окон показывают ускорение активности перед назначением
    for short_window, long_window in [("1d", "7d"), ("3d", "14d"), ("7d", "30d")]:
        short_col = f"event_count_{short_window}"
        long_col = f"event_count_{long_window}"
        if short_col in features.columns and long_col in features.columns:
            features[f"event_recent_share_{short_window}_{long_window}"] = (
                    features[short_col] / (features[long_col] + 1.0)
            )
    # Разность вложенных окон оценивает активность только в промежутке между их границами
    ordered_event_windows = list(EVENT_WINDOWS_SECONDS)
    for short_window, long_window in zip(ordered_event_windows[:-1], ordered_event_windows[1:]):
        short_col = f"event_count_{short_window}"
        long_col = f"event_count_{long_window}"
        if short_col in features.columns and long_col in features.columns:
            features[f"event_count_bucket_{short_window}_{long_window}"] = (
                features[long_col] - features[short_col]
            ).clip(lower=0)
            short_seconds = EVENT_WINDOWS_SECONDS[short_window]
            long_seconds = EVENT_WINDOWS_SECONDS[long_window]
            features[f"event_velocity_{short_window}_{long_window}"] = (
                features[short_col] / short_seconds
            ) / (features[long_col] / long_seconds + 1e-5)
    if "event_high_intent_count" in features.columns and "event_count_all" in features.columns:
        features["event_high_intent_share"] = (
                features["event_high_intent_count"] / (features["event_count_all"] + 1.0)
        )

    features = features.reset_index().rename(columns={"index": "lead_id"})
    # Левый джойн сохраняет обращения, для которых допустимых событий не найдено
    all_leads = assignments[["lead_id"]].drop_duplicates()
    features = all_leads.merge(features, on="lead_id", how="left", validate="one_to_one")
    zero_prefixes = (
        "event_count_",
        "event_decay_",
        "event_transition_count_",
        "event_ctx_seq_count_",
        "event_src_slot_count_",
        "event_type_ctx_seq_count_",
        "event_type_src_slot_count_",
        "event_session_count",
        "event_type_nunique",
        "event_src_slot_nunique",
        "event_ctx_seq_nunique",
        "event_stage_progressions",
        "event_stage_regressions",
        "event_high_intent_count",
    )
    for col in features.columns:
        if col.startswith(zero_prefixes):
            features[col] = features[col].fillna(0.0)
    # Отдельный индикатор различает отсутствие истории и нулевую активность
    if "event_count_all" in features.columns:
        features["event_has_history"] = (features["event_count_all"] > 0).astype("int8")
    return reduce_numeric_memory(features)


## Признаки на основе готовых табличных агрегатов


In [ ]:
# Формирует скорости, доли и изменения активности между временными окнами
# Считает отношение с добавлением сглаживания в знаменатель
def safe_ratio(numerator: pd.Series, denominator: pd.Series, smoothing: float = 1.0) -> pd.Series:
    return numerator.astype("float64") / (denominator.astype("float64") + smoothing)


# Строит скорости, доли и изменения активности между временными окнами
def add_window_features(df: pd.DataFrame) -> pd.DataFrame:
    # Группировка метрик по всем доступным временным окнам
    # Регексом объединяем колонки одной метрики с разными временными окнами
    window_pattern = re.compile(r"^(.*)_(1d|3d|7d|14d|30d|90d)$")
    families: dict[str, dict[str, str]] = {}
    original_columns = list(df.columns)
    for col in original_columns:
        match = window_pattern.match(col)
        if match and pd.api.types.is_numeric_dtype(df[col]):
            base, window = match.groups()
            families.setdefault(base, {})[window] = col

    # Рассчитываем масштабы отдельных окон и сравниваем соседние окна
    new_features: dict[str, pd.Series] = {}
    for base, mapping in families.items():
        for window, col in mapping.items():
            days = WINDOW_DAYS[window]
            values = df[col].astype("float64")
            # Скорость приводит накопленные значения окон разной длины к общей дневной шкале
            new_features[f"{base}_rate_{window}"] = values / days
            new_features[f"{base}_log1p_{window}"] = np.log1p(values.clip(lower=0))

        for short, long in zip(WINDOW_ORDER[:-1], WINDOW_ORDER[1:]):
            if short not in mapping or long not in mapping:
                continue
            short_col = mapping[short]
            long_col = mapping[long]
            short_days = WINDOW_DAYS[short]
            long_days = WINDOW_DAYS[long]
            short_values = df[short_col].astype("float64")
            long_values = df[long_col].astype("float64")
            short_rate = short_values / short_days
            # Из длинного окна вычитается вложенное короткое окно для оценки предшествующего периода
            prior_window_rate = (long_values - short_values).clip(lower=0) / (long_days - short_days)
            new_features[f"{base}_delta_{short}_{long}"] = long_values - short_values
            new_features[f"{base}_prior_rate_{short}_{long}"] = prior_window_rate
            new_features[f"{base}_share_{short}_{long}"] = safe_ratio(short_values, long_values, 1.0)
            new_features[f"{base}_velocity_{short}_{long}"] = safe_ratio(short_rate, long_values / long_days, 1e-3)
            new_features[f"{base}_accel_{short}_{long}"] = safe_ratio(short_rate, prior_window_rate, 1e-3)

    # Добавляем колонки
    if new_features:
        df = pd.concat([df, pd.DataFrame(new_features, index=df.index)], axis=1)
    return df


# Формирует признаки времени, цены, истории назначений и пользовательской активности
# Строит признаки времени, цены, автомобиля, продавца и истории обращений
def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    new_features: dict[str, pd.Series] = {}

    # Применяем тригонометрические преобразования к периодическим признакам времени.
    # Это позволяет модели корректно обрабатывать цикличность временных промежутков.
    if "assignment_hour" in df.columns:
        hour = df["assignment_hour"].astype("float64")
        new_features["assignment_hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
        new_features["assignment_hour_cos"] = np.cos(2 * np.pi * hour / 24.0)
    if "assignment_weekday" in df.columns:
        weekday = df["assignment_weekday"].astype("float64")
        new_features["assignment_weekday_sin"] = np.sin(2 * np.pi * weekday / 7.0)
        new_features["assignment_weekday_cos"] = np.cos(2 * np.pi * weekday / 7.0)

    # Суммарная вовлеченность объединяет действия разной глубины в общий объем активности
    engagement_components = [
        "item_views",
        "item_favorites",
        "detail_expands",
        "photo_swipes",
        "seller_page_views",
        "search_views",
        "query_refinements",
        "similar_item_clicks",
        "saved_search_matches",
        "user_contacts",
        "chat_opens",
        "call_clicks",
    ]
    for window in WINDOW_ORDER:
        available = [f"{name}_{window}" for name in engagement_components if f"{name}_{window}" in df.columns]
        if available:
            new_features[f"engagement_total_{window}"] = df[available].sum(axis=1, min_count=1)

        contact_cols = [c for c in [f"user_contacts_{window}", f"chat_opens_{window}", f"call_clicks_{window}"] if
                        c in df.columns]
        if contact_cols:
            new_features[f"contact_intent_{window}"] = df[contact_cols].sum(axis=1, min_count=1)

        if f"item_views_{window}" in df.columns:
            views = df[f"item_views_{window}"]
            for action in ["item_favorites", "detail_expands", "photo_swipes", "user_contacts", "chat_opens",
                           "call_clicks"]:
                action_col = f"{action}_{window}"
                if action_col in df.columns:
                    new_features[f"{action}_per_view_{window}"] = safe_ratio(df[action_col], views, 1.0)

        # Исторические конверсии описывают прохождение прошлых обращений по этапам воронки
        assigned = f"leadgen_prev_assigned_{window}"
        answered = f"leadgen_prev_answered_{window}"
        positive = f"leadgen_prev_positive_{window}"
        if assigned in df.columns and answered in df.columns:
            new_features[f"leadgen_answer_rate_{window}"] = safe_ratio(df[answered], df[assigned], 1.0)
        if assigned in df.columns and positive in df.columns:
            new_features[f"leadgen_positive_rate_{window}"] = safe_ratio(df[positive], df[assigned], 1.0)
        if answered in df.columns and positive in df.columns:
            new_features[f"leadgen_positive_per_answer_{window}"] = safe_ratio(df[positive], df[answered], 1.0)

    # Связь продавец-пользователь-автомобиль
    if "seller_response_rate_30d" in df.columns and "seller_inventory_count" in df.columns:
        new_features["seller_response_inventory_interaction"] = (
                df["seller_response_rate_30d"].astype("float64")
                * np.log1p(df["seller_inventory_count"].clip(lower=0).astype("float64"))
        )
    if "user_active_days_30d" in df.columns and "user_age_days" in df.columns:
        new_features["user_activity_density"] = safe_ratio(df["user_active_days_30d"], df["user_age_days"], 1.0)
    if "prior_assignments_30d" in df.columns and "user_active_days_30d" in df.columns:
        new_features["assignments_per_active_day"] = safe_ratio(
            df["prior_assignments_30d"], df["user_active_days_30d"], 1.0
        )
    if "car_age_years" in df.columns and "mileage_km_log" in df.columns:
        new_features["mileage_log_per_car_age"] = safe_ratio(
            df["mileage_km_log"], df["car_age_years"].clip(lower=0), 1.0
        )

    # Делим активность, действия с высокой значимостью и контакты
    for window in WINDOW_ORDER:
        engagement_col = f"engagement_total_{window}"
        contact_col = f"contact_intent_{window}"
        if engagement_col in new_features and contact_col in new_features:
            new_features[f"contact_share_of_engagement_{window}"] = safe_ratio(
                new_features[contact_col], new_features[engagement_col], 1.0
            )
        search_cols = [
            c for c in [f"search_views_{window}", f"query_refinements_{window}", f"similar_item_clicks_{window}"]
            if c in df.columns
        ]
        if search_cols:
            new_features[f"search_intent_{window}"] = df[search_cols].sum(axis=1, min_count=1)
        high_intent_cols = [
            c for c in
            [f"item_favorites_{window}", f"user_contacts_{window}", f"chat_opens_{window}", f"call_clicks_{window}"]
            if c in df.columns
        ]
        if high_intent_cols:
            new_features[f"high_intent_total_{window}"] = df[high_intent_cols].sum(axis=1, min_count=1)

    # Проверка соогласованности накопительных окон и структуры пропусков
    window_pattern = re.compile(r"^(.*)_(1d|3d|7d|14d|30d|90d)$")
    families: dict[str, dict[str, str]] = {}
    for col in df.columns:
        match = window_pattern.match(col)
        if match and pd.api.types.is_numeric_dtype(df[col]):
            families.setdefault(match.group(1), {})[match.group(2)] = col
    # Накопительный счетчик длинного окна не должен быть меньше счетчика вложенного окна
    violation_terms = []
    missing_terms = []
    for mapping in families.values():
        ordered = [mapping[w] for w in WINDOW_ORDER if w in mapping]
        if len(ordered) >= 2:
            values = df[ordered]
            violation_terms.append((values.diff(axis=1).iloc[:, 1:] < 0).sum(axis=1))
            missing_terms.append(values.isna().sum(axis=1))
    if violation_terms:
        new_features["window_monotonicity_violations"] = pd.concat(violation_terms, axis=1).sum(axis=1)
        new_features["window_family_missing_count"] = pd.concat(missing_terms, axis=1).sum(axis=1)

    # Общая структура пропусков может отражать доступность источников данных для обращения
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        new_features["row_missing_numeric_count"] = df[numeric_cols].isna().sum(axis=1).astype("float32")
        new_features["row_missing_numeric_fraction"] = df[numeric_cols].isna().mean(axis=1).astype("float32")

    # Присоединяем новые признаки 
    if new_features:
        df = pd.concat([df, pd.DataFrame(new_features, index=df.index)], axis=1)
    return df


# Формирует сочетания источника, региона, канала, сегмента и времени назначения
# Объединяет связанные категориальные признаки в парные сочетания
def add_categorical_crosses(df: pd.DataFrame) -> pd.DataFrame:
    pairs = [
        ("region", "car_segment"),
        ("region", "price_bucket"),
        ("region", "lead_source"),
        ("region", "call_center"),
        ("lead_source", "lead_channel"),
        ("lead_source", "call_center"),
        ("lead_source", "car_segment"),
        ("lead_source", "user_tenure_bucket"),
        ("lead_channel", "car_segment"),
        ("call_center", "lead_channel"),
        ("user_tenure_bucket", "lead_channel"),
        ("price_bucket", "car_segment"),
        ("price_bucket", "lead_source"),
        ("assignment_weekday", "assignment_hour"),
    ]
    # Парные значения позволяют деревьям различать локальные сочетания категорий
    for left, right in pairs:
        if left in df.columns and right in df.columns:
            left_values = df[left].astype("string").fillna(MISSING_CATEGORY)
            right_values = df[right].astype("string").fillna(MISSING_CATEGORY)
            df[f"cross_{left}_{right}"] = left_values + "__" + right_values
    return df


# Сопоставляет готовые табличные агрегаты с активностью из журнала событий
# Сопоставляет табличную активность с агрегатами журнала событий
def add_post_event_features(df: pd.DataFrame) -> pd.DataFrame:
    new_features: dict[str, pd.Series] = {}
    # Сравниваем агрегаты с готовыми табличными показателями
    if "item_price_log" in df.columns and "event_price_mean" in df.columns:
        new_features["event_price_mean_diff"] = df["item_price_log"] - df["event_price_mean"]
    if "item_price_log" in df.columns and "event_last_price_log" in df.columns:
        new_features["event_last_price_diff"] = df["item_price_log"] - df["event_last_price_log"]
    if "event_count_7d" in df.columns and "item_views_7d" in df.columns:
        new_features["event_to_tabular_views_7d"] = safe_ratio(df["event_count_7d"], df["item_views_7d"], 1.0)
    if "event_count_30d" in df.columns and "item_views_30d" in df.columns:
        new_features["event_to_tabular_views_30d"] = safe_ratio(df["event_count_30d"], df["item_views_30d"], 1.0)
    if "event_session_count" in df.columns and "event_count_all" in df.columns:
        new_features["event_events_per_session"] = safe_ratio(
            df["event_count_all"], df["event_session_count"], 0.5
        )
    if "event_recency_sec" in df.columns:
        new_features["event_recency_log1p"] = np.log1p(df["event_recency_sec"].clip(lower=0))
        for horizon_hours in [1, 6, 12, 24, 72, 168]:
            new_features[f"event_recency_le_{horizon_hours}h"] = (
                df["event_recency_sec"] <= horizon_hours * 3600.0
            ).astype("float32")

    # Одноименные действия из двух источников сравниваются в одинаковых временных окнах
    event_to_table = {
        "item_view": "item_views",
        "favorite": "item_favorites",
        "search": "search_views",
        "chat_open": "chat_opens",
        "call_click": "call_clicks",
    }
    for event_type, table_base in event_to_table.items():
        for window in ["1d", "3d", "7d", "14d", "30d"]:
            event_col = f"event_count_{window}_type_{event_type}"
            table_col = f"{table_base}_{window}"
            if event_col in df.columns and table_col in df.columns:
                new_features[f"event_table_ratio_{event_type}_{window}"] = safe_ratio(
                    df[event_col], df[table_col], 1.0
                )
    # Динамика воронки
    if "event_stage_last" in df.columns and "event_stage_max" in df.columns:
        new_features["event_stage_drop_from_peak"] = df["event_stage_max"] - df["event_stage_last"]
    if "event_stage_progressions" in df.columns and "event_stage_regressions" in df.columns:
        new_features["event_stage_net_progress"] = (
                df["event_stage_progressions"] - df["event_stage_regressions"]
        )
    if new_features:
        df = pd.concat([df, pd.DataFrame(new_features, index=df.index)], axis=1)
    return df


# Объединяет признаки и приводит колонки к форматам моделей
def prepare_features(
        train: pd.DataFrame,
        test: pd.DataFrame,
        event_features: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str]]:
    # Нормализуем формат временных полей
    train = train.copy()
    test = test.copy()

    train["assignment_ts"] = pd.to_datetime(train["assignment_ts"], errors="coerce")
    test["assignment_ts"] = pd.to_datetime(test["assignment_ts"], errors="coerce")
    train["assignment_date"] = pd.to_datetime(train["assignment_date"], errors="coerce").dt.normalize()
    test["assignment_date"] = pd.to_datetime(test["assignment_date"], errors="coerce").dt.normalize()

    # Хронологический порядок необходим для последующего построения временных фолдов
    train = train.sort_values(["assignment_ts", "lead_id"], kind="mergesort").reset_index(drop=True)

    train = add_window_features(train)
    test = add_window_features(test)
    train = add_domain_features(train)
    test = add_domain_features(test)
    train = add_categorical_crosses(train)
    test = add_categorical_crosses(test)

    train = train.merge(event_features, on="lead_id", how="left", validate="one_to_one")
    test = test.merge(event_features, on="lead_id", how="left", validate="one_to_one")

    LOGGER.info("Добавление признаков сравнения активности")
    train = add_post_event_features(train)
    test = add_post_event_features(test)

    # В модель попадают только колонки, которые присутствуют одновременно в train и test
    feature_columns = [
        col for col in train.columns
        if col not in NON_FEATURE_COLUMNS and col in test.columns
    ]

    # Отделяем категориальные признаки
    categorical_columns = [
        col for col in feature_columns
        if (
                pd.api.types.is_object_dtype(train[col])
                or pd.api.types.is_string_dtype(train[col])
                or isinstance(train[col].dtype, pd.CategoricalDtype)
        )
    ]

    LOGGER.info("Обработка категориальных признаков count=%d", len(categorical_columns))

    # Частотное кодирование выполняется совместно на обучающей и тестовой выборке
    # для корректного сопоставления редких категорий и снижения риска переобучения.
    for col in categorical_columns:
        combined = pd.concat(
            [
                train[col].astype("string").fillna(MISSING_CATEGORY),
                test[col].astype("string").fillna(MISSING_CATEGORY),
            ],
            ignore_index=True,
        )
        # Общий словарь категорий гарантирует одинаковые внутренние коды в обеих выборках
        categories = pd.Index(combined.astype(str).unique(), dtype=object)
        train_values = train[col].astype("string").fillna(MISSING_CATEGORY).astype(str)
        test_values = test[col].astype("string").fillna(MISSING_CATEGORY).astype(str)
        train[col] = pd.Categorical(train_values, categories=categories)
        test[col] = pd.Categorical(test_values, categories=categories)

        # Частота категории добавляется как числовая оценка ее распространенности
        frequencies = combined.value_counts(dropna=False) / len(combined)
        train[f"freq_{col}"] = train_values.map(frequencies).astype("float32")
        test[f"freq_{col}"] = test_values.map(frequencies).astype("float32")
        feature_columns.append(f"freq_{col}")

    numeric_columns = [c for c in feature_columns if c not in categorical_columns]
    for col in numeric_columns:
        train[col] = pd.to_numeric(train[col], errors="coerce").astype("float32")
        test[col] = pd.to_numeric(test[col], errors="coerce").astype("float32")

    return train, test, feature_columns, categorical_columns


## Метрика и хронологическая валидация


In [ ]:
# Вычисляет Average Precision для одного набора обращений
def average_precision_binary(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y = np.asarray(y_true, dtype=np.int8)
    scores = np.asarray(y_score, dtype=np.float64)
    positives = int(y.sum())
    # День без положительных обращений получает нулевое значение метрики
    if positives == 0:
        return 0.0

    order = np.argsort(scores, kind="mergesort")[::-1]
    y_sorted = y[order]
    scores_sorted = scores[order]
    # Метрика обновляется на границах уникальных score, чтобы одинаковые значения считались совместно
    distinct_end = np.where(np.diff(scores_sorted))[0]
    threshold_end = np.r_[distinct_end, len(scores_sorted) - 1]
    true_positives = np.cumsum(y_sorted)[threshold_end]
    precision = true_positives / (threshold_end + 1.0)
    recall = true_positives / positives
    return float(np.sum(np.diff(np.r_[0.0, recall]) * precision))


# Усредняет Average Precision по датам назначения
def daily_average_precision(
        y_true: np.ndarray | pd.Series,
        y_score: np.ndarray | pd.Series,
        dates: np.ndarray | pd.Series,
) -> float:
    y = np.asarray(y_true)
    scores = np.asarray(y_score)
    date_values = pd.Series(dates).astype("string").to_numpy()
    result: list[float] = []
    for date in pd.unique(date_values):
        mask = date_values == date
        result.append(average_precision_binary(y[mask], scores[mask]))
    return float(np.mean(result)) if result else 0.0


# Преобразует прогнозы в относительные ранги внутри каждой даты
def daywise_rank(scores: np.ndarray, dates: pd.Series | np.ndarray) -> np.ndarray:
    frame = pd.DataFrame({"score": np.asarray(scores), "date": pd.Series(dates).astype("string").to_numpy()})
    grouped = frame.groupby("date", sort=False)["score"]
    ranks = grouped.rank(method="average", ascending=True)
    counts = grouped.transform("size").astype("float64")
    result = np.where(counts.to_numpy() > 1.0, (ranks.to_numpy() - 1.0) / (counts.to_numpy() - 1.0), 0.5)
    return result.astype("float64")


# Создает хронологические разбиения со скользящим окном
def make_rolling_date_folds(
        dates: pd.Series,
        n_splits: int,
        valid_days: int = 2,
        min_train_days: int = 8,
) -> list[tuple[np.ndarray, np.ndarray]]:
    date_series = pd.to_datetime(dates).dt.normalize()
    unique_dates = np.array(sorted(date_series.dropna().unique()))
    if len(unique_dates) < min_train_days + valid_days:
        min_train_days = max(1, len(unique_dates) - valid_days)
    if len(unique_dates) < 2 or valid_days < 1:
        raise ValueError("Недостаточно уникальных дат для формирования хронологической валидации")

    # Начало валидации сдвигается вперед, а обучение всегда содержит только более ранние даты
    starts = list(range(min_train_days, len(unique_dates) - valid_days + 1, valid_days))
    if not starts:
        starts = [len(unique_dates) - valid_days]
    # Берутся последние разбиения, поскольку они ближе к периоду скрытого теста
    starts = starts[-max(1, n_splits):]

    folds: list[tuple[np.ndarray, np.ndarray]] = []
    for start in starts:
        valid_dates = unique_dates[start: start + valid_days]
        valid_start = pd.Timestamp(valid_dates[0])
        # Строгое неравенство не допускает попадания дня валидации в обучение
        train_mask = date_series < valid_start
        valid_mask = date_series.isin(valid_dates)
        train_idx = np.flatnonzero(train_mask.to_numpy())
        valid_idx = np.flatnonzero(valid_mask.to_numpy())
        if len(train_idx) and len(valid_idx):
            folds.append((train_idx, valid_idx))
    if not folds:
        raise ValueError("Не удалось сформировать валидационные разбиения")
    return folds


# Оставляет записи из последних дней обучающего периода
def restrict_to_recent_dates(
        indices: np.ndarray,
        dates: pd.Series,
        recent_days: int,
) -> np.ndarray:
    if recent_days <= 0 or len(indices) == 0:
        return indices
    selected_dates = pd.to_datetime(dates.iloc[indices]).dt.normalize()
    max_date = selected_dates.max()
    cutoff = max_date - pd.Timedelta(days=recent_days - 1)
    recent_mask = selected_dates >= cutoff
    recent_indices = indices[recent_mask.to_numpy()]
    # Ограничение отменяется, если после него остается слишком мало строк для устойчивого обучения
    return recent_indices if len(recent_indices) >= 500 else indices


# Выравнивает вклад дат и усиливает вес недавних наблюдений
def training_weights(
        dates: pd.Series | np.ndarray,
        recency_half_life_days: float = 0.0,
) -> np.ndarray:
    date_series = pd.Series(dates).astype("string")
    counts = date_series.value_counts()
    # Обратная частота даты выравнивает суммарный вес каждого дня
    weights = date_series.map(lambda d: 1.0 / counts[d]).to_numpy(dtype=np.float64)
    if recency_half_life_days > 0:
        parsed = pd.to_datetime(date_series).dt.normalize()
        age_days = (parsed.max() - parsed).dt.days.to_numpy(dtype=np.float64)
        # Экспоненциальное затухание уменьшает вес в два раза за заданный период
        weights *= np.exp(-math.log(2.0) * age_days / recency_half_life_days)
    return weights / weights.mean()


# Преобразует категориальные признаки в строковый формат для CatBoost
def make_catboost_frame(X: pd.DataFrame, categorical_columns: list[str]) -> pd.DataFrame:
    result = X.copy()
    for col in categorical_columns:
        result[col] = result[col].astype("string").fillna(MISSING_CATEGORY).astype(str)
    return result


## Модели


In [ ]:
# Возвращает фиксированные служебные параметры LightGBM
def base_lgb_params(seed: int) -> dict[str, Any]:
    return {
        "objective": "binary",
        "metric": "None",
        "bagging_freq": 1,
        "min_gain_to_split": 0.0,
        "max_bin": 255,
        "verbosity": -1,
        "seed": seed,
        "feature_fraction_seed": seed,
        "bagging_seed": seed,
        "data_random_seed": seed,
        "device_type": MODEL_DEVICES["lgb"],
        "num_threads": CPU_THREADS,
    }


# Возвращает фиксированные служебные параметры XGBoost
def base_xgb_params(seed: int) -> dict[str, Any]:
    return {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "device": MODEL_DEVICES["xgb"],
        "enable_categorical": True,
        "max_cat_threshold": 64,
        "seed": seed,
        "nthread": CPU_THREADS,
    }


# Возвращает фиксированные служебные параметры CatBoost
def base_cat_params(seed: int) -> dict[str, Any]:
    params: dict[str, Any] = {
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
        "bootstrap_type": "Bayesian",
        "border_count": 128,
        "boosting_type": "Plain",
        "random_seed": seed,
        "task_type": MODEL_DEVICES["cat"],
        "allow_writing_files": False,
        "verbose": False,
    }
    if MODEL_DEVICES["cat"] == "GPU":
        params["devices"] = "0"
    else:
        params["thread_count"] = CPU_THREADS
    return params


# Обучает LightGBM и определяет число итераций по временной валидации
def fit_lgb_classifier(
        X_train: pd.DataFrame,
        y_train: pd.Series,
        dates_train: pd.Series,
        categorical_columns: list[str],
        params: dict[str, Any],
        num_boost_round: int,
        X_valid: pd.DataFrame | None = None,
        y_valid: pd.Series | None = None,
        dates_valid: pd.Series | None = None,
        early_stopping_rounds: int = 120,
        recency_half_life_days: float = 0.0,
) -> tuple[lgb.Booster, int]:
    # Передаём LighGBM-у веса и признаки
    train_set = lgb.Dataset(
        X_train,
        label=y_train,
        weight=training_weights(dates_train, recency_half_life_days),
        categorical_feature=categorical_columns,
        free_raw_data=False,
    )
    # Без валидации используется заранее выбранное число итераций финального обучения
    if X_valid is None or y_valid is None or dates_valid is None:
        model = lgb.train(params, train_set, num_boost_round=num_boost_round)
        return model, num_boost_round

    valid_set = lgb.Dataset(
        X_valid,
        label=y_valid,
        categorical_feature=categorical_columns,
        reference=train_set,
        free_raw_data=False,
    )
    
    valid_dates_array = pd.Series(dates_valid).to_numpy()

    # Передает Daily Average Precision в цикл обучения LightGBM
    def feval(preds: np.ndarray, dataset: lgb.Dataset) -> tuple[str, float, bool]:
        return "daily_ap", daily_average_precision(dataset.get_label(), preds, valid_dates_array), True

    # Ранняя остановка по DAP
    model = lgb.train(
        params,
        train_set,
        num_boost_round=num_boost_round,
        valid_sets=[valid_set],
        valid_names=["valid"],
        feval=feval,
        callbacks=[
            lgb.early_stopping(early_stopping_rounds, first_metric_only=True, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )
    best_iteration = model.best_iteration or num_boost_round
    return model, int(best_iteration)


# Получает прогноз LightGBM на заданном числе итераций
def predict_lgb(model: lgb.Booster, X: pd.DataFrame, iteration: int) -> np.ndarray:
    return model.predict(X, num_iteration=iteration)


# Обучает XGBoost и определяет число итераций по временной валидации
def fit_xgb_classifier(
        X_train: pd.DataFrame,
        y_train: pd.Series,
        dates_train: pd.Series,
        params: dict[str, Any],
        num_boost_round: int,
        X_valid: pd.DataFrame | None = None,
        y_valid: pd.Series | None = None,
        dates_valid: pd.Series | None = None,
        early_stopping_rounds: int = 120,
        recency_half_life_days: float = 0.0,
) -> tuple[xgb.Booster, int]:
    # DMatrix сохраняет категориальные типы pandas для нативных разбиений XGBoost
    # Передаём веса и признаки в XGBoost
    dtrain = xgb.DMatrix(
        X_train,
        label=y_train,
        weight=training_weights(dates_train, recency_half_life_days),
        enable_categorical=True,
    )
    if X_valid is None or y_valid is None or dates_valid is None:
        model = xgb.train(params, dtrain, num_boost_round=num_boost_round, verbose_eval=False)
        return model, num_boost_round

    # Блок валидации создает отдельную DMatrix и сохраняет даты для вычисления DAP
    dvalid = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)
    valid_dates_array = pd.Series(dates_valid).to_numpy()

    # Передает DAP в цикл обучения XGBoost
    def custom_metric(preds: np.ndarray, dmatrix: xgb.DMatrix) -> tuple[str, float]:
        return "daily_ap", daily_average_precision(dmatrix.get_label(), preds, valid_dates_array)

    # Максимизируем DAP, забираем ещё и лучшую итерацию
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        evals=[(dvalid, "valid")],
        custom_metric=custom_metric,
        maximize=True,
        early_stopping_rounds=early_stopping_rounds,
        verbose_eval=False,
    )
    best_iteration = int(getattr(model, "best_iteration", num_boost_round - 1)) + 1
    return model, best_iteration


# Получает прогноз XGBoost на заданном числе итераций
def predict_xgb(model: xgb.Booster, X: pd.DataFrame, iteration: int) -> np.ndarray:
    dmatrix = xgb.DMatrix(X, enable_categorical=True)
    return model.predict(dmatrix, iteration_range=(0, max(1, iteration)))


# Обучает CatBoost
def fit_cat_classifier(
        X_train: pd.DataFrame,
        y_train: pd.Series,
        dates_train: pd.Series,
        categorical_columns: list[str],
        params: dict[str, Any],
        iterations: int,
        X_valid: pd.DataFrame | None = None,
        y_valid: pd.Series | None = None,
        early_stopping_rounds: int = 120,
        recency_half_life_days: float = 0.0,
) -> tuple[CatBoostClassifier, int]:
    # Pool передает CatBoost индексы категориальных признаков и веса строк без ручного кодирования
    train_pool = Pool(
        X_train,
        y_train,
        cat_features=categorical_columns,
        weight=training_weights(dates_train, recency_half_life_days),
    )
    model = CatBoostClassifier(**params, iterations=iterations)
    if X_valid is None or y_valid is None:
        model.fit(train_pool, verbose=False)
        return model, iterations

    # Блок валидации включает раннюю остановку и возврат фактического числа деревьев
    valid_pool = Pool(X_valid, y_valid, cat_features=categorical_columns)
    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        early_stopping_rounds=early_stopping_rounds,
        verbose=False,
    )
    best_iteration = model.get_best_iteration()
    if best_iteration is None or best_iteration < 0:
        best_iteration = iterations - 1
    return model, int(best_iteration) + 1


# Получает вероятность положительного класса от CatBoost
def predict_cat(model: CatBoostClassifier, X: pd.DataFrame) -> np.ndarray:
    return model.predict_proba(X)[:, 1]


# Задает пространство поиска гиперпараметров Optuna
def suggest_params(trial: optuna.Trial, model_name: str) -> dict[str, Any]:
    # Управление параметрами LighGBM вроде глубины дерева
    if model_name == "lgb":
        return {
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.06, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 24, 128),
            "max_depth": trial.suggest_int("max_depth", 5, 10),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 160),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.55, 0.95),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.55, 0.95),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-4, 5.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 0.1, 20.0, log=True),
            "cat_smooth": trial.suggest_float("cat_smooth", 5.0, 60.0),
            "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.8, 4.5, log=True),
        }
    # Аналогичный блок для XGBoost
    if model_name == "xgb":
        return {
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.06, log=True),
            "max_depth": trial.suggest_int("max_depth", 4, 9),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 15.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.55, 0.95),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 0.95),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 3.0),
            "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 2, 12),
            "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.8, 4.5, log=True),
        }
    # И для CatBoost (причём как для обычного, так и для обучающегося на недавнем периоде)
    if model_name in {"cat", "cat_recent"}:
        return {
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.07, log=True),
            "depth": trial.suggest_int("depth", 5, 9),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 25.0, log=True),
            "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
            "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.8, 4.5, log=True),
        }
    raise KeyError(model_name)



# Выбирает фиксированные параметры выбранной модели
def fixed_model_params(model_name: str, seed: int) -> dict[str, Any]:
    if model_name == "lgb":
        return base_lgb_params(seed)
    if model_name == "xgb":
        return base_xgb_params(seed)
    if model_name in {"cat", "cat_recent"}:
        return base_cat_params(seed)
    raise KeyError(model_name)




# Обучает одну модель на одном временном разбиении
def fit_one_model(
        model_name: str,
        X_model: pd.DataFrame,
        y: pd.Series,
        dates: pd.Series,
        train_idx: np.ndarray,
        valid_idx: np.ndarray,
        categorical_columns: list[str],
        params: dict[str, Any],
        max_rounds: int,
        early_stopping_rounds: int,
        recent_days: int,
        recency_half_life_days: float,
) -> tuple[Any, np.ndarray, int]:
    if model_name == "lgb":
        model, iteration = fit_lgb_classifier(
            X_model.iloc[train_idx], y.iloc[train_idx], dates.iloc[train_idx],
            categorical_columns, params, max_rounds,
            X_model.iloc[valid_idx], y.iloc[valid_idx], dates.iloc[valid_idx],
            early_stopping_rounds, recency_half_life_days,
        )
        return model, predict_lgb(model, X_model.iloc[valid_idx], iteration), iteration

    if model_name == "xgb":
        model, iteration = fit_xgb_classifier(
            X_model.iloc[train_idx], y.iloc[train_idx], dates.iloc[train_idx],
            params, max_rounds, X_model.iloc[valid_idx], y.iloc[valid_idx],
            dates.iloc[valid_idx], early_stopping_rounds, recency_half_life_days,
        )
        return model, predict_xgb(model, X_model.iloc[valid_idx], iteration), iteration
   
    if model_name in {"cat", "cat_recent"}:
        effective_train_idx = train_idx
        # cat_recent обучается только на последних датах текущего обучающего фолда
        if model_name == "cat_recent":
            effective_train_idx = restrict_to_recent_dates(train_idx, dates, recent_days)
        model, iteration = fit_cat_classifier(
            X_model.iloc[effective_train_idx], y.iloc[effective_train_idx],
            dates.iloc[effective_train_idx], categorical_columns, params, max_rounds,
            X_model.iloc[valid_idx], y.iloc[valid_idx], early_stopping_rounds,
            recency_half_life_days,
        )
        return model, predict_cat(model, X_model.iloc[valid_idx]), iteration
    raise KeyError(model_name)


## Настройка параметров и ансамбль


In [ ]:
# Подбирает гиперпараметры модели по временным разбиениям
def tune_model(
        model_name: str,
        X_model: pd.DataFrame,
        y: pd.Series,
        dates: pd.Series,
        categorical_columns: list[str],
        folds: list[tuple[np.ndarray, np.ndarray]],
        trials: int,
        max_rounds: int,
        early_stopping_rounds: int,
        recency_half_life_days: float,
) -> dict[str, Any]:
    # Делим поиск и фиксированные параметры
    fixed = fixed_model_params(model_name, RANDOM_STATE)

    # Возвращает Optuna среднее значение DAP
    def objective(trial: optuna.Trial) -> float:
        # Объединяем параметры и создаем накопители результатов фолдов
        trial_started = time.perf_counter()
        params = {**fixed, **suggest_params(trial, model_name)}
        scores: list[float] = []
        best_iterations: list[int] = []
        LOGGER.info("Tune start model=%s trial=%d/%d", model_name, trial.number + 1, trials)
        # Проверяем комбинацию на фолдах
        for fold_id, (train_idx, valid_idx) in enumerate(folds):
            fold_started = time.perf_counter()
            LOGGER.info(
                "Tune fold start model=%s trial=%d/%d fold=%d/%d",
                model_name, trial.number + 1, trials, fold_id + 1, len(folds),
            )
            model, pred, best_iteration = fit_one_model(
                model_name, X_model, y, dates, train_idx, valid_idx,
                categorical_columns, params, max_rounds, early_stopping_rounds,
                recent_days=0, recency_half_life_days=recency_half_life_days,
            )
            score = daily_average_precision(y.iloc[valid_idx], pred, dates.iloc[valid_idx])
            scores.append(score)
            best_iterations.append(best_iteration)
            trial.report(float(np.mean(scores)), step=fold_id)
            del model, pred
            gc.collect()
            # Неэффективная попытка может быть остановлена между фолдами
            if trial.should_prune():
                LOGGER.info(
                    "Tune pruned model=%s trial=%d/%d folds=%d seconds=%.1f",
                    model_name, trial.number + 1, trials, fold_id + 1,
                    time.perf_counter() - trial_started,
                )
                raise optuna.TrialPruned()

        # Агрегируем качество и число итераций по временным фолдам попытки
        mean_score = float(np.mean(scores))
        LOGGER.info(
            "Tune done model=%s trial=%d/%d daily_ap=%.6f median_round=%d seconds=%.1f",
            model_name, trial.number + 1, trials, mean_score,
            int(np.median(best_iterations)), time.perf_counter() - trial_started,
        )
        return mean_score

    # TPE-поиском ищем лучший конфиг гиперпараметров
    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_STATE, multivariate=True),
    )
    study.optimize(objective, n_trials=trials, n_jobs=1, show_progress_bar=False)
    return {**fixed, **study.best_params}



# Собирает прогнозы вне выборки для всех моделей ансамбля
def cross_validate_models(
        model_names: list[str],
        X: pd.DataFrame,
        X_cat: pd.DataFrame,
        y: pd.Series,
        dates: pd.Series,
        categorical_columns: list[str],
        folds: list[tuple[np.ndarray, np.ndarray]],
        params_by_model: dict[str, dict[str, Any]],
        max_rounds: int,
        early_stopping_rounds: int,
        recent_days: int,
        recency_half_life_days: float,
) -> tuple[dict[str, np.ndarray], dict[str, list[int]], dict[str, float]]:
    # NaN отмечает строки, которые не входили ни в один валидационный период
    # OOF-массивы и история лучших итераций
    oof = {name: np.full(len(X), np.nan, dtype=np.float64) for name in model_names}
    iterations = {name: [] for name in model_names}

    # Ходим и учимся по фолдам
    for fold_id, (train_idx, valid_idx) in enumerate(folds, start=1):
        LOGGER.info("Кросс-валидация fold=%d/%d", fold_id, len(folds))
        fold_scores: list[str] = []
        for model_name in model_names:
            LOGGER.info("Обучение модели model=%s fold=%d/%d", model_name, fold_id, len(folds))
            X_model = X_cat if model_name in {"cat", "cat_recent"} else X
            model, pred, best_iteration = fit_one_model(
                model_name, X_model, y, dates, train_idx, valid_idx,
                categorical_columns, params_by_model[model_name], max_rounds,
                early_stopping_rounds, recent_days, recency_half_life_days,
            )
            # Прогноз записывается только для строк, не использованных при обучении этой модели
            oof[model_name][valid_idx] = pred
            iterations[model_name].append(best_iteration)
            score = daily_average_precision(y.iloc[valid_idx], pred, dates.iloc[valid_idx])
            fold_scores.append(f"{model_name}={score:.6f}")
            del model, pred
            collect_garbage(f"cv_{fold_id}_{model_name}")
        LOGGER.info("Кросс-валидация завершена fold=%d/%d scores=%s", fold_id, len(folds), " ".join(fold_scores))

    # Сравнение моделей и подбор смеси выполняются на общем наборе OOF-строк
    common_mask = np.logical_and.reduce([np.isfinite(oof[name]) for name in model_names])
    model_scores = {
        name: daily_average_precision(y[common_mask], oof[name][common_mask], dates[common_mask])
        for name in model_names
    }
    return oof, iterations, model_scores



# Подбирает веса дневных рангов моделей по прогнозам вне выборки
def optimize_blend_weights(
        oof: dict[str, np.ndarray],
        y: pd.Series,
        dates: pd.Series,
        trials: int,
        seed: int,
        shrinkage: float,
) -> tuple[dict[str, float], float, np.ndarray]:
    # Конвертируем прогнозы в ранги в рамках дня
    model_names = list(oof)
    mask = np.logical_and.reduce([np.isfinite(oof[name]) for name in model_names])
    y_valid = y[mask].to_numpy()
    dates_valid = dates[mask].reset_index(drop=True)
    rank_matrix = np.column_stack([
        daywise_rank(oof[name][mask], dates_valid) for name in model_names
    ])

    n_models = len(model_names)
    equal = np.full(n_models, 1.0 / n_models)
    # В набор кандидатов заранее включаются равная смесь, одиночные модели и все пары
    candidates: list[np.ndarray] = [equal]
    candidates.extend(np.eye(n_models))
    for i in range(n_models):
        for j in range(i + 1, n_models):
            pair = np.zeros(n_models)
            pair[i] = pair[j] = 0.5
            candidates.append(pair)

    # Добавляет в поиск выборки из распределения Дирихле (для единичной суммы)
    rng = np.random.default_rng(seed)
    for alpha in [0.5, 1.0, 2.0, 5.0]:
        count = max(1, trials // 4)
        candidates.extend(rng.dirichlet(np.full(n_models, alpha), size=count))

    # Считаем качество с устойчивостью
    unique_dates = pd.Series(dates_valid).astype("string").unique()
    best_objective = -np.inf
    best_weights = equal.copy()
    for weights in candidates:
        blend = rank_matrix @ weights
        day_scores = []
        date_values = pd.Series(dates_valid).astype("string").to_numpy()
        for date in unique_dates:
            day_mask = date_values == date
            day_scores.append(average_precision_binary(y_valid[day_mask], blend[day_mask]))
        mean_score = float(np.mean(day_scores))
        # Штрафы уменьшают предпочтение нестабильных по дням и чрезмерно концентрированных смесей
        stability_penalty = 0.02 * float(np.std(day_scores))
        concentration_penalty = 0.002 * float(np.square(weights - equal).sum())
        objective = mean_score - stability_penalty - concentration_penalty
        if objective > best_objective:
            best_objective = objective
            best_weights = weights.copy()

    # Сжимаем лучшую смесь, считаем её прогноз
    shrinkage = float(np.clip(shrinkage, 0.0, 1.0))
    final_weights = (1.0 - shrinkage) * best_weights + shrinkage * equal
    final_weights /= final_weights.sum()
    final_blend = rank_matrix @ final_weights
    final_score = daily_average_precision(y_valid, final_blend, dates_valid)

    # Восстанавливаем индексацию и возвращаем результат
    weights_dict = {name: float(weight) for name, weight in zip(model_names, final_weights)}
    blended_oof = np.full(len(y), np.nan, dtype=np.float64)
    blended_oof[mask] = final_blend
    return weights_dict, float(final_score), blended_oof


## Финальное обучение и формирование результата


In [ ]:
# Определяет число итераций финальной модели по результатам валидации
def median_final_iterations(iterations: list[int], max_rounds: int) -> int:
    # Небольшой запас компенсирует обучение финальной модели на большем объеме данных
    value = int(round(float(np.median(iterations)) * 1.08))
    minimum = min(50, max_rounds)
    return max(minimum, min(max_rounds, value))



# Обучаем финальную модель и получаем прогноз тестовой выборки
def fit_final_model(
        model_name: str,
        X_model: pd.DataFrame,
        X_test_model: pd.DataFrame,
        y: pd.Series,
        dates: pd.Series,
        categorical_columns: list[str],
        params: dict[str, Any],
        iterations: int,
        seed: int,
        recent_days: int,
        recency_half_life_days: float,
) -> tuple[Any, np.ndarray]:
    # Копия защищает общие найденные параметры от изменения при переборе сидов
    seeded_params = params.copy()
    # Учим LightGBM
    if model_name == "lgb":
        seeded_params.update(
            seed=seed,
            feature_fraction_seed=seed,
            bagging_seed=seed,
            data_random_seed=seed,
        )
        model, _ = fit_lgb_classifier(
            X_model, y, dates, categorical_columns, seeded_params, iterations,
            recency_half_life_days=recency_half_life_days,
        )
        return model, predict_lgb(model, X_test_model, iterations)
    # И XGBoost
    if model_name == "xgb":
        seeded_params["seed"] = seed
        model, _ = fit_xgb_classifier(
            X_model, y, dates, seeded_params, iterations,
            recency_half_life_days=recency_half_life_days,
        )
        return model, predict_xgb(model, X_test_model, iterations)
    # И CatBoost
    if model_name in {"cat", "cat_recent"}:
        seeded_params["random_seed"] = seed
        effective_idx = np.arange(len(X_model))
        if model_name == "cat_recent":
            effective_idx = restrict_to_recent_dates(effective_idx, dates, recent_days)
        model, _ = fit_cat_classifier(
            X_model.iloc[effective_idx], y.iloc[effective_idx], dates.iloc[effective_idx],
            categorical_columns, seeded_params, iterations,
            recency_half_life_days=recency_half_life_days,
        )
        return model, predict_cat(model, X_test_model)
    raise KeyError(model_name)



# Извлекает важность признаков из обученной модели
def extract_importance(model: Any, model_name: str, feature_columns: list[str]) -> pd.DataFrame:
    if model_name == "lgb":
        values = model.feature_importance(importance_type="gain")
        names = model.feature_name()
        return pd.DataFrame({"feature": names, "importance": values, "model": model_name})
    # XGBoost возвращает словарь только для использованных признаков, остальные получают нулевую важность
    if model_name == "xgb":
        mapping = model.get_score(importance_type="gain")
        values = [mapping.get(name, 0.0) for name in feature_columns]
        return pd.DataFrame({"feature": feature_columns, "importance": values, "model": model_name})
    if model_name in {"cat", "cat_recent"}:
        values = model.get_feature_importance()
        return pd.DataFrame({"feature": feature_columns, "importance": values, "model": model_name})
    raise KeyError(model_name)


# Вычисляет идентификатор конфигурации запуска
def stable_config_hash(config: RunConfig) -> str:
    payload = json.dumps(asdict(config), sort_keys=True).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:12]


In [ ]:
# Выполняет полный цикл подготовки данных, обучения и создания результата
def run(config: RunConfig) -> None:
    output_dir = Path(config.output_dir)
    configure_logging(output_dir)
    LOGGER.info("Старт config_hash=%s peak_ram=%.2fGB", stable_config_hash(config), memory_gb())
    LOGGER.info("Вычислительные ресурсы lightgbm=%s xgboost=%s catboost=%s", MODEL_DEVICES["lgb"], MODEL_DEVICES["xgb"], MODEL_DEVICES["cat"])

    LOGGER.info("Загрузка данных")
    train_raw, test_raw, events, sample_submission = load_data(config.data_dir)
    validate_input(train_raw, test_raw, events)
    LOGGER.info("Размерности данных train=%s test=%s events=%s", train_raw.shape, test_raw.shape, events.shape)

    LOGGER.info("Генерация признаков на основе событий")
    event_features = build_event_features(train_raw, test_raw, events)
    collect_garbage("event_features")

    LOGGER.info("Подготовка и преобразование признаков")
    train, test, feature_columns, categorical_columns = prepare_features(
        train_raw, test_raw, event_features
    )
    # После объединения тяжелые исходные таблицы освобождаются до начала обучения
    matched_events = int(event_features["event_count_all"].gt(0).sum())
    del train_raw, test_raw, events, event_features
    collect_garbage("prepared_features")
    LOGGER.info("Итоговые размерности features=%d categorical=%d matched_events=%d", len(feature_columns), len(categorical_columns), matched_events)

    # Получаем признаки, таргет, фреймы для кетбуста
    X = train[feature_columns]
    X_test = test[feature_columns]
    y = train[TARGET].astype("int8")
    dates = train["assignment_date"]
    X_cat = make_catboost_frame(X, categorical_columns)
    X_test_cat = make_catboost_frame(X_test, categorical_columns)

    # Отдельные временные разбиения используются для Optuna и для итоговых OOF-прогнозов
    # Строим фолды для тюнинга и OOF
    tune_folds = make_rolling_date_folds(
        dates,
        config.tune_folds,
        valid_days=config.valid_days,
        min_train_days=config.min_train_days,
    )
    cv_folds = make_rolling_date_folds(
        dates,
        config.cv_folds,
        valid_days=config.valid_days,
        min_train_days=config.min_train_days,
    )
    model_names = ["lgb", "xgb", "cat", "cat_recent"]

    # Подбор гиперпараметров
    params_by_model: dict[str, dict[str, Any]] = {}
    for model_name in ["cat", "lgb", "xgb"]:
        LOGGER.info("Оптимизация параметров model=%s trials=%d", model_name, config.tune_trials)
        LOGGER.info(
            "Tune plan model=%s trials=%d folds=%d max_rounds=%d device=%s",
            model_name, config.tune_trials, len(tune_folds), config.max_rounds,
            MODEL_DEVICES[model_name],
        )
        params_by_model[model_name] = tune_model(
            model_name,
            X_cat if model_name == "cat" else X,
            y,
            dates,
            categorical_columns,
            tune_folds,
            config.tune_trials,
            config.max_rounds,
            config.early_stopping_rounds,
            config.recency_half_life_days,
        )
        collect_garbage(f"tune_{model_name}")
    params_by_model["cat_recent"] = params_by_model["cat"].copy()


    oof, iterations_by_model, model_scores = cross_validate_models(
        model_names,
        X,
        X_cat,
        y,
        dates,
        categorical_columns,
        cv_folds,
        params_by_model,
        config.max_rounds,
        config.early_stopping_rounds,
        config.recent_days,
        config.recency_half_life_days,
    )

    # Веса ансамбля подбираются только по прогнозам вне обучающих частей фолдов
    blend_weights, blend_score, blended_oof = optimize_blend_weights(
        oof,
        y,
        dates,
        config.blend_trials,
        RANDOM_STATE,
        config.blend_shrinkage,
    )
    LOGGER.info("Daily AP на валидации scores=%s", " ".join(f"{k}={v:.6f}" for k, v in model_scores.items()))
    LOGGER.info("Результат ансамбля score=%.6f weights=%s", blend_score, json.dumps(blend_weights, sort_keys=True))

    # Финальный конфиг
    final_iterations = {
        name: median_final_iterations(iterations_by_model[name], config.max_rounds)
        for name in model_names
    }
    seeds = [RANDOM_STATE + 997 * i for i in range(config.final_seeds)]
    test_predictions: dict[str, np.ndarray] = {
        name: np.zeros(len(test), dtype=np.float64) for name in model_names
    }
    importance_frames: list[pd.DataFrame] = []

    # Финальное обучение
    LOGGER.info("Обучение финальных моделей")
    for model_name in model_names:

        LOGGER.info("Финальная модель model=%s rounds=%d seeds=%d", model_name, final_iterations[model_name], len(seeds))
        for seed_id, seed in enumerate(seeds):
            LOGGER.info("Финальное обучение model=%s seed=%d/%d value=%d", model_name, seed_id + 1, len(seeds), seed)
            model, pred = fit_final_model(
                model_name,
                X_cat if model_name in {"cat", "cat_recent"} else X,
                X_test_cat if model_name in {"cat", "cat_recent"} else X_test,
                y,
                dates,
                categorical_columns,
                params_by_model[model_name],
                final_iterations[model_name],
                seed,
                config.recent_days,
                config.recency_half_life_days,
            )
            # Усреднение вероятностей нескольких сидов выполняется без хранения всех прогнозов
            test_predictions[model_name] += pred / len(seeds)
            if seed_id == 0:
                importance_frames.append(extract_importance(model, model_name, feature_columns))
            del model
            collect_garbage(f"final_{model_name}_{seed_id + 1}")

    # Та же дневная ранговая нормализация применяется к тесту перед смешиванием моделей
    test_rank_matrix = np.column_stack([
        daywise_rank(test_predictions[name], test["assignment_date"])
        for name in model_names
    ])
    weights_array = np.array([blend_weights[name] for name in model_names], dtype=np.float64)
    blended_test = test_rank_matrix @ weights_array
    final_scores = daywise_rank(blended_test, test["assignment_date"])

    # Создаём сабмишн
    submission = pd.DataFrame(
        {
            "lead_id": test["lead_id"].astype(str),
            "score": np.clip(final_scores, 0.0, 1.0),
        }
    )

    #На всякий восстанавливаем порядок строк

    expected_ids = sample_submission["lead_id"].astype(str)
    if set(expected_ids) != set(submission["lead_id"]):
        raise AssertionError("Идентификаторы теста и примера отправки не совпадают")
    submission = expected_ids.to_frame().merge(submission, on="lead_id", how="left")

    # Проверяем формат всего и вся
    if list(submission.columns) != ["lead_id", "score"]:
        raise AssertionError("Колонки выходного файла должны содержать строго lead_id и score.")
    if len(submission) != len(test) or not submission["lead_id"].is_unique:
        raise AssertionError("Неверное количество строк или идентификаторы lead_id дублируются.")
    if submission["score"].isna().any() or not submission["score"].between(0, 1).all():
        raise AssertionError("Оценки должны быть числовыми и находиться в диапазоне от 0 до 1.")

    # Сохраняемся
    submission.to_csv(output_dir / "submission.csv", index=False)

    oof_frame = train[["lead_id", "assignment_date", TARGET]].copy()
    for name in model_names:
        oof_frame[f"pred_{name}"] = oof[name]
    oof_frame["pred_blend"] = blended_oof
    oof_frame.to_csv(output_dir / "oof_predictions.csv", index=False)

    importance = pd.concat(importance_frames, ignore_index=True)
    importance.to_csv(output_dir / "feature_importance.csv", index=False)

    metadata = {
        "config": asdict(config),
        "config_hash": stable_config_hash(config),
        "devices": MODEL_DEVICES,
        "feature_count": len(feature_columns),
        "categorical_count": len(categorical_columns),
        "models": model_names,
        "params": params_by_model,
        "cv_scores": model_scores,
        "blend_score": blend_score,
        "blend_weights": blend_weights,
        "final_iterations": final_iterations,
        "seeds": seeds,
    }
    with open(output_dir / "run_metadata.json", "w", encoding="utf-8") as file:
        json.dump(metadata, file, ensure_ascii=False, indent=2, default=str)
    with open(output_dir / "feature_columns.json", "w", encoding="utf-8") as file:
        json.dump(feature_columns, file, ensure_ascii=False, indent=2)

    LOGGER.info("Output %s", output_dir / "submission.csv")


## Запуск

Ячейка выполняет полный цикл обучения и сохраняет следующие файлы:

- `submission.csv`
- `oof_predictions.csv`
- `feature_importance.csv`
- `run_metadata.json`
- `feature_columns.json`
- `training.log`


In [ ]:
run(CONFIG)


19:08:39Z | INFO | Старт config_hash=c26ffe6097f6 peak_ram=0.67GB
19:08:39Z | INFO | Вычислительные ресурсы lightgbm=cpu xgboost=cuda catboost=GPU
19:08:39Z | INFO | Загрузка данных
19:08:40Z | INFO | Размерности данных train=(13694, 119) test=(4306, 118) events=(254705, 7)
19:08:40Z | INFO | Генерация признаков на основе событий
19:08:51Z | INFO | GC event_features           objects=0 peak_ram=1.22GB
19:08:51Z | INFO | Подготовка и преобразование признаков
19:08:52Z | INFO | Добавление признаков сравнения активности
19:08:53Z | INFO | Обработка категориальных признаков count=33
19:08:54Z | INFO | GC prepared_features        objects=0 peak_ram=1.28GB
19:08:54Z | INFO | Итоговые размерности features=1322 categorical=33 matched_events=17977
19:08:54Z | INFO | Оптимизация параметров model=cat trials=40
19:08:54Z | INFO | Tune plan model=cat trials=40 folds=5 max_rounds=2400 device=GPU
19:08:54Z | INFO | Tune start model=cat trial=1/40
19:08:54Z | INFO | Tune fold start model=cat trial=1/4

In [13]:
submission_path = Path(CONFIG.output_dir) / "submission.csv"
submission_check = pd.read_csv(submission_path)
assert list(submission_check.columns) == ["lead_id", "score"]
assert submission_check["lead_id"].is_unique
assert submission_check["score"].between(0, 1).all()
submission_check.head()


,lead_id,score
0,lead_97e409eb8f8c8246,0.826835
1,lead_55310edb4489f9e9,0.613240
2,lead_e7f653a2c6a7eee8,0.959862
3,lead_22f8e1cfc487ac20,0.118119
4,lead_48b638b839abfac3,0.391876
